# ML-10 — Content Action Playbook

This notebook turns the grouped-validation model into a ranked content review queue.

Each page receives a model score, one reason code, and one suggested action.

The queue is a decision-support tool. A human must review every page before making changes.

In [1]:
import os
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import hf_hub_download
from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
TOKEN = userdata.get("HF_TOKEN")

if not TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

REPO = "FlyRank/internship-warehouse"

FACT = hf_hub_download(
    REPO,
    "fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=TOKEN
)

DIM_CONTENT = hf_hub_download(
    REPO,
    "dim_content.parquet",
    repo_type="dataset",
    token=TOKEN
)

con = duckdb.connect()

print("Warehouse files loaded successfully.")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Warehouse files loaded successfully.


In [3]:
df = con.sql(f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_early,
        SUM(gsc_clicks) AS clicks_early,
        AVG(gsc_avg_position) AS avg_position_early,

        SUM(gsc_impressions) FILTER (
            WHERE report_date <= DATE '2026-03-10'
        ) AS imp_h1,

        SUM(gsc_impressions) FILTER (
            WHERE report_date > DATE '2026-03-10'
        ) AS imp_h2

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
    GROUP BY 1, 2
),

late AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_late

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'
    GROUP BY 1, 2
)

SELECT
    e.client_hash_id,
    e.content_hash_id,
    e.impressions_early,

    e.clicks_early * 1.0 /
        NULLIF(e.impressions_early, 0) AS ctr_early,

    e.avg_position_early,

    (COALESCE(e.imp_h2, 0) + 1.0) /
        (COALESCE(e.imp_h1, 0) + 1.0) AS impr_trend_early,

    DATE '2026-03-20' - c.content_updated_date
        AS days_since_update,

    c.search_volume,

    (
        COALESCE(l.clicks_late, 0) / 11.0
        < 0.8 * (e.clicks_early / 20.0)
    ) AS label

FROM early e

JOIN read_parquet('{DIM_CONTENT}') c
    USING (client_hash_id, content_hash_id)

LEFT JOIN late l
    USING (client_hash_id, content_hash_id)

WHERE e.clicks_early >= 5
  AND c.content_updated_date <= DATE '2026-03-20'

""").df()

df["label"] = df["label"].astype(int)
df["days_since_update"] = df["days_since_update"].astype(float)
df["search_volume"] = df["search_volume"].astype(float)

print("Dataset shape:", df.shape)

Dataset shape: (2526, 9)


In [4]:
df["search_volume"] = df["search_volume"].fillna(
    df["search_volume"].median()
)

df["impressions_log"] = np.log1p(
    df["impressions_early"]
)

df["search_volume_log"] = np.log1p(
    df["search_volume"]
)

df["impr_trend_log"] = np.log(
    df["impr_trend_early"].clip(0.05, 20)
)

FEATS = [
    "impressions_log",
    "ctr_early",
    "avg_position_early",
    "days_since_update",
    "search_volume_log",
    "impr_trend_log"
]

LABEL = "label"
GROUP = "client_hash_id"

d = df.dropna(
    subset=[LABEL, GROUP]
).copy().reset_index(drop=True)

print("Final dataset shape:", d.shape)
print("Positive label rate:", round(d[LABEL].mean(), 3))

Final dataset shape: (2526, 12)
Positive label rate: 0.59


In [5]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=0
)

tr_i, te_i = next(
    gss.split(d, groups=d[GROUP])
)

tr = d.iloc[tr_i].copy()
te = d.iloc[te_i].copy()

m = make_pipeline(
    SimpleImputer(strategy="median"),

    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        random_state=0,
        n_jobs=-1
    )
)

m.fit(
    tr[FEATS],
    tr[LABEL]
)

te["model_score"] = m.predict_proba(
    te[FEATS]
)[:, 1]


def precision_at_k(y, scores, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y)[order].mean())


model_p50 = precision_at_k(
    te[LABEL],
    te["model_score"],
    50
)

base_rate = te[LABEL].mean()

print("Training clients:", tr[GROUP].nunique())
print("Testing clients:", te[GROUP].nunique())
print("Overlapping clients:", len(
    set(tr[GROUP]) & set(te[GROUP])
))

print("Grouped test rows:", len(te))
print("Grouped test base rate:", round(base_rate, 3))
print("Model Precision@50:", round(model_p50, 3))

Training clients: 15
Testing clients: 7
Overlapping clients: 0
Grouped test rows: 1400
Grouped test base rate: 0.536
Model Precision@50: 0.62


In [6]:
volume_part = 1 - te["impressions_early"].rank(pct=True)

stale_part = te["days_since_update"].rank(pct=True)

W_VOL = 0.8
W_STALE = 0.2

te["baseline_score"] = (
    W_VOL * volume_part
    + W_STALE * stale_part
)

baseline_p50 = precision_at_k(
    te[LABEL],
    te["baseline_score"],
    50
)

comparison = pd.DataFrame({
    "Method": [
        "Week 4 rule baseline",
        "Random Forest model"
    ],
    "Precision@50": [
        round(baseline_p50, 3),
        round(model_p50, 3)
    ],
    "Test base rate": [
        round(base_rate, 3),
        round(base_rate, 3)
    ]
})

display(comparison)

,Method,Precision@50,Test base rate
0,Week 4 rule baseline,0.82,0.536
1,Random Forest model,0.62,0.536


## 1. Ranked Actions and Reason Codes

The queue ranks pages by the grouped-split model score. Each page receives one reason code based on its strongest relative signal among three selected features.

The reason code explains which signal is most unusual compared with other pages in the test set. It does not explain the model internally and does not prove why a page received its label.

| Reason code | What it means | Suggested action | Archetype |
|---|---|---|---|
| STALE_CONTENT | The page was updated longer ago than most pages in the test set. | Check accuracy and freshness; refresh only if needed. | Improve |
| LOW_VOLUME_DECLINE_RISK | The page had relatively low early impressions. | Review page performance and check whether the click data is noisy. | Monitor |
| FALLING_VISIBILITY | The early impressions trend was relatively low. | Review the impressions trend and possible demand changes. | Review |

### Decay and refresh insight

In the Week 4 sample, the 61+ days bucket had a proxy-label rate of 0.765 (n=34), compared with 0.587 for pages updated within 0–23 days (n=2,399).

This is a directional observation only: the oldest bucket is small, and age alone does not prove that a page needs refreshing.

Week 4 also showed that lower-volume pages carried the proxy label more often in this sample. CTR did not show a useful pattern across its buckets, so I do not use CTR as a reason code.

In [7]:
# Direction-adjusted percentile signals.
# Higher values mean a stronger reason signal.

reason_strengths = pd.DataFrame({
    "STALE_CONTENT":
        te["days_since_update"].rank(pct=True),

    "LOW_VOLUME_DECLINE_RISK":
        1 - te["impressions_early"].rank(pct=True),

    "FALLING_VISIBILITY":
        1 - te["impr_trend_early"].rank(pct=True)
}, index=te.index)

te["reason_code"] = reason_strengths.idxmax(axis=1)

ACTION_MAP = {
    "STALE_CONTENT":
        "Check freshness and accuracy",

    "LOW_VOLUME_DECLINE_RISK":
        "Review page performance and data volume",

    "FALLING_VISIBILITY":
        "Review impressions trend and search demand"
}

REVIEW_NOTE = {
    "STALE_CONTENT":
        "Age alone does not prove the content is outdated.",

    "LOW_VOLUME_DECLINE_RISK":
        "Low impressions can make click changes noisy.",

    "FALLING_VISIBILITY":
        "A short-term impressions dip may reflect seasonality or demand."
}

te["action"] = te["reason_code"].map(ACTION_MAP)
te["review_note"] = te["reason_code"].map(REVIEW_NOTE)

queue = (
    te.sort_values("model_score", ascending=False)
    [["content_hash_id", "model_score", "reason_code", "action"]]
    .rename(columns={"content_hash_id": "page_id"})
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

queue = queue[
    ["rank", "page_id", "model_score", "reason_code", "action"]
]

display(queue.head(10))

,rank,page_id,model_score,reason_code,action
0,1,content_21edb58fbbd89201,0.824433,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume
1,2,content_fd3bc29fcb17d66c,0.792016,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume
2,3,content_0bf56604dc25c03a,0.788402,FALLING_VISIBILITY,Review impressions trend and search demand
3,4,content_e17810d5b5318415,0.783417,FALLING_VISIBILITY,Review impressions trend and search demand
4,5,content_4676aa4ec15f166c,0.781917,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume
5,6,content_8b674d827d5da401,0.778556,FALLING_VISIBILITY,Review impressions trend and search demand
6,7,content_451abfa90cae4639,0.776108,STALE_CONTENT,Check freshness and accuracy
7,8,content_bbe6f8bbdbc99b08,0.774017,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume
8,9,content_0be20b39bc0622ca,0.773403,FALLING_VISIBILITY,Review impressions trend and search demand
9,10,content_7d9bef4a837ec178,0.773181,FALLING_VISIBILITY,Review impressions trend and search demand


In [8]:
STALE = "days_since_update"

d["stale_bucket"] = pd.cut(
    d[STALE],
    bins=[-1, 23, 60, 400],
    labels=["0-23 days", "24-60 days", "61+ days"]
)

decay = (
    d.groupby("stale_bucket", observed=True)[LABEL]
    .agg(n="size", rate="mean")
    .round(3)
)

display(decay)

,n,rate
stale_bucket,,
0-23 days,2399,0.587
24-60 days,93,0.591
61+ days,34,0.765


In [9]:
review_top20 = (
    te.sort_values("model_score", ascending=False)
    .head(20)
    .copy()
)

review_top20["review_note"] = (
    review_top20["reason_code"].map(REVIEW_NOTE)
)

review_table = review_top20[
    [
        "content_hash_id",
        "model_score",
        "reason_code",
        "action",
        "impressions_early",
        "days_since_update",
        "impr_trend_early",
        "review_note"
    ]
].rename(columns={"content_hash_id": "page_id"})

display(review_table.round(3))

,page_id,model_score,reason_code,action,impressions_early,days_since_update,impr_trend_early,review_note
690,content_21edb58fbbd89201,0.824,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume,380.0,23.0,0.475,Low impressions can make click changes noisy.
2388,content_fd3bc29fcb17d66c,0.792,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume,166.0,23.0,0.474,Low impressions can make click changes noisy.
525,content_0bf56604dc25c03a,0.788,FALLING_VISIBILITY,Review impressions trend and search demand,3962.0,23.0,0.529,A short-term impressions dip may reflect seaso...
1803,content_e17810d5b5318415,0.783,FALLING_VISIBILITY,Review impressions trend and search demand,6614.0,23.0,0.752,A short-term impressions dip may reflect seaso...
1166,content_4676aa4ec15f166c,0.782,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume,437.0,23.0,0.663,Low impressions can make click changes noisy.
1922,content_8b674d827d5da401,0.779,FALLING_VISIBILITY,Review impressions trend and search demand,7845.0,23.0,0.595,A short-term impressions dip may reflect seaso...
1008,content_451abfa90cae4639,0.776,STALE_CONTENT,Check freshness and accuracy,4631.0,25.0,2.335,Age alone does not prove the content is outdated.
724,content_bbe6f8bbdbc99b08,0.774,LOW_VOLUME_DECLINE_RISK,Review page performance and data volume,394.0,23.0,0.445,Low impressions can make click changes noisy.
522,content_0be20b39bc0622ca,0.773,FALLING_VISIBILITY,Review impressions trend and search demand,4984.0,23.0,0.468,A short-term impressions dip may reflect seaso...
2431,content_7d9bef4a837ec178,0.773,FALLING_VISIBILITY,Review impressions trend and search demand,7344.0,23.0,0.729,A short-term impressions dip may reflect seaso...


## 2. Intended Use and Limits

### Intended use

The output is a ranked review queue for a content team. It helps reviewers decide which pages to inspect first.

The model score is a ranking signal, not a confirmed probability that a page needs a refresh.

### Limits

- The target is a proxy based on click decline, not a verified content-refresh decision.
- The model was evaluated using one grouped split from one month of data.
- The grouped test set is highly imbalanced across clients, so performance may not generalize to other clients.
- The reason codes describe selected feature signals, not proven causes of decline.
- Low-volume pages may have noisy click measurements.
- The model does not know the page's business importance, search intent, factual accuracy, or current editorial plans.
- Scores may change when search demand, content, or data collection changes.

The queue is for human review and prioritization only. It is not an automated publishing or deletion system.

## 3. Human Review and the No-Go List

### Human review rules

- A person reviews every page before any change is made.
- The reviewer checks the page's purpose, search intent, accuracy, and recent history.
- Low-volume pages receive extra scrutiny because their click data may be noisy.
- A reviewer can reject any recommendation that does not make sense.
- Rejected rows should be logged with a reason to improve future versions.

### Cost and value assumptions

For this prototype, I use a working review capacity of 50 pages per cycle.

I estimate that reviewing a false positive costs about 10 minutes. This is an operational assumption, not a measured team average.

A missed page may delay identifying a content issue, but I do not have enough evidence to assign it a monetary cost.

The code below compares the model and baseline in terms of expected proxy-label hits among the top 50.

### What should NOT be automated

- Publishing, rewriting, or deleting a page based on its score alone.
- Merging or pruning pages without human review.
- Treating a proxy label as proof that a page needs a refresh.
- Reporting the score as proof of ranking impact.
- Applying the model to clients or periods outside the evaluation without further validation.

In [10]:
REVIEW_CAPACITY = 50
FALSE_POSITIVE_REVIEW_MIN = 10

model_hits = model_p50 * REVIEW_CAPACITY
baseline_hits = baseline_p50 * REVIEW_CAPACITY

model_false_positives = REVIEW_CAPACITY - model_hits

print("Review capacity per cycle:", REVIEW_CAPACITY)

print(
    "Model expected proxy-label hits:",
    round(model_hits, 1),
    "out of 50"
)

print(
    "Baseline expected proxy-label hits:",
    round(baseline_hits, 1),
    "out of 50"
)

print(
    "Model expected false positives:",
    round(model_false_positives, 1)
)

print(
    "Estimated false-positive review time:",
    round(model_false_positives * FALSE_POSITIVE_REVIEW_MIN, 1),
    "minutes"
)

print(
    "Grouped test base rate:",
    round(base_rate, 3)
)

Review capacity per cycle: 50
Model expected proxy-label hits: 31.0 out of 50
Baseline expected proxy-label hits: 41.0 out of 50
Model expected false positives: 19.0
Estimated false-positive review time: 190.0 minutes
Grouped test base rate: 0.536


## 4. Monitoring and Retrain Triggers

The following checks are proposed for each new labelled cycle.

| Check | Trigger | Response |
|---|---|---|
| Precision@50 on the newest labelled month | Falls below the grouped rule baseline measured in this notebook | Pause use and investigate |
| Feature distributions | Median changes by more than 20% from the development month, where the baseline median is non-zero | Review data and features |
| Reviewer rejection rate in the top 50 | Above 30% | Revisit reason codes and the label |
| New data fields or site changes | Any change | Repeat the leakage and feature-timing audit |

The 20% and 30% thresholds are initial monitoring choices, not statistically validated limits.

Retrain only when a trigger appears in two consecutive cycles. Use a fresh grouped split when evaluating the updated model.

Do not treat retraining as an automatic fix for poor data quality or a changing target definition.

## 5. Exports for the Paper

The notebook exports:

- `work/outputs/action_queue.csv` — the ranked queue, regenerated on each run and not committed.
- `work/outputs/w07_metrics.json` — the baseline and model Precision@50 receipt.
- `work/figures/model_vs_baseline.png` — comparison of model and rule baseline.
- `work/figures/decay_by_staleness.png` — proxy-label rate by staleness bucket.

The metrics receipt and figures are intended to be committed with the notebook.

In [11]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Export ranked queue.
# Keep this generated queue out of git.

queue.to_csv(
    "work/outputs/action_queue.csv",
    index=False
)


# Figure 1: Model vs baseline

comparison_plot = pd.DataFrame({
    "Method": [
        "Rule baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

ax = comparison_plot.plot.bar(
    x="Method",
    y="Precision@50",
    legend=False,
    rot=0,
    figsize=(6, 4)
)

ax.set_ylabel("Precision@50")
ax.set_title("Model vs baseline — grouped test split")

plt.tight_layout()

plt.savefig(
    "work/figures/model_vs_baseline.png",
    dpi=150
)

plt.close()


# Figure 2: Decay insight

decay["rate"].plot.bar(
    figsize=(6, 4),
    rot=30
)

plt.ylabel("Share with proxy label")
plt.title("Proxy-label rate by staleness bucket")

plt.tight_layout()

plt.savefig(
    "work/figures/decay_by_staleness.png",
    dpi=150
)

plt.close()


# Metrics receipt

metrics = {
    "baseline_precision_at_50": round(float(baseline_p50), 3),
    "model_precision_at_50": round(float(model_p50), 3),
    "grouped_test_base_rate": round(float(base_rate), 3),
    "n_test_rows": int(len(te)),
    "split": "grouped_by_client"
}

with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exports completed.")
print(metrics)

Exports completed.
{'baseline_precision_at_50': 0.82, 'model_precision_at_50': 0.62, 'grouped_test_base_rate': 0.536, 'n_test_rows': 1400, 'split': 'grouped_by_client'}


## 6. Self-check

- [x] Intended use and limitations explained.
- [x] Ranked action queue created.
- [x] Each page receives one reason code and one suggested action.
- [x] Reason codes use the Week 4 findings and selected feature signals.
- [x] Staleness buckets and decay insight included.
- [x] Model compared with the rule baseline on the same grouped test split.
- [x] Human review rules and no-go list included.
- [x] Cost and value assumptions documented.
- [x] Monitoring and retrain triggers defined.
- [x] Queue, figures, and metrics receipt exported.
- [ ] Notebook runs from top to bottom without errors.
- [ ] No client names, domains, URLs, or tokens appear in outputs.
- [ ] Notebook committed under `work/notebooks/w07_action_playbook.ipynb`.

In [12]:
required_files = [
    "work/outputs/action_queue.csv",
    "work/outputs/w07_metrics.json",
    "work/figures/model_vs_baseline.png",
    "work/figures/decay_by_staleness.png"
]

print("Checking exported files:")

for file in required_files:
    print(file, "->", os.path.exists(file))


# Check that the exported queue does not expose client identifiers.

assert "client_hash_id" not in queue.columns
assert "domain" not in queue.columns
assert "url" not in queue.columns
assert "token" not in queue.columns

print("\nQueue privacy check passed.")
print("Queue rows:", len(queue))
print("Queue columns:", queue.columns.tolist())

Checking exported files:
work/outputs/action_queue.csv -> True
work/outputs/w07_metrics.json -> True
work/figures/model_vs_baseline.png -> True
work/figures/decay_by_staleness.png -> True

Queue privacy check passed.
Queue rows: 1400
Queue columns: ['rank', 'page_id', 'model_score', 'reason_code', 'action']
